# Ch01-10: 실전: 금융 거래 데이터 종합 분석 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

포괄적 EDA + Benford 검정

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from datetime import datetime, timedelta

np.random.seed(42)
n=20000; nc=500; base=datetime(2025,1,1)
cids = np.random.randint(1,nc+1,n)
dates = [base+timedelta(days=int(d)) for d in np.cumsum(np.random.geometric(0.3,n))][:n]
amounts = np.random.lognormal(4,1.5,n).round(2)
fi = np.random.choice(n,int(n*0.02),replace=False); amounts[fi]*=np.random.uniform(5,20,len(fi))
cats = np.random.choice(['식료품','교통','쇼핑','외식','온라인','금융','의료'],n,p=[.25,.15,.20,.15,.10,.10,.05])
df = pd.DataFrame({'date':pd.to_datetime(dates),'cid':cids,'amount':amounts,'cat':cats,'fraud':0})
df.loc[fi,'fraud']=1

# 시간 패턴
df['dow'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

fig,axes = plt.subplots(2,2,figsize=(14,10))
df.groupby('dow')['amount'].count().plot(kind='bar',ax=axes[0,0],title='요일별 거래 수')
df.groupby('cat')['amount'].mean().sort_values().plot(kind='barh',ax=axes[0,1],title='카테고리별 평균 금액')
axes[1,0].hist(np.log10(df['amount']+1), bins=50, alpha=0.7); axes[1,0].set_title('log10(금액) 분포')

# Benford
first_digits = df['amount'].apply(lambda x: int(str(abs(x)).replace('.','').lstrip('0')[0]) if x>0 else 0)
first_digits = first_digits[first_digits>0]
observed = first_digits.value_counts(normalize=True).sort_index()
expected = pd.Series({d: np.log10(1+1/d) for d in range(1,10)})
axes[1,1].bar(observed.index-0.15, observed.values, 0.3, label='관측', alpha=0.7)
axes[1,1].bar(expected.index+0.15, expected.values, 0.3, label='Benford', alpha=0.7)
axes[1,1].set_title("Benford's Law 검정"); axes[1,1].legend()
plt.tight_layout(); plt.show()

chi2, p = stats.chisquare(observed.values*len(first_digits), expected.values*len(first_digits))
print(f"Benford χ² 검정: stat={chi2:.2f}, p={p:.4f}")


**결과 해석**: 거래 금액이 Benford's law와 크게 괴리되면 조작/사기 가능성 시사. 이상 거래 주입으로 분포가 약간 변형된 것을 확인할 수 있다.

---
## 문제 2 풀이

RFM 분석 + K-means 세분화

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from datetime import datetime, timedelta

np.random.seed(42)
n=20000; nc=500; base=datetime(2025,1,1)
cids = np.random.randint(1,nc+1,n)
dates = [base+timedelta(days=int(d)) for d in np.cumsum(np.random.geometric(0.3,n))][:n]
amounts = np.random.lognormal(4,1.5,n).round(2)
fi = np.random.choice(n,int(n*0.02),replace=False); amounts[fi]*=np.random.uniform(5,20,len(fi))
df = pd.DataFrame({'date':pd.to_datetime(dates),'cid':cids,'amount':amounts,'fraud':0})
df.loc[fi,'fraud']=1

ref_date = df['date'].max()
rfm = df.groupby('cid').agg(
    R=('date', lambda x: (ref_date-x.max()).days),
    F=('amount', 'count'),
    M=('amount', 'sum'),
    fraud_rate=('fraud', 'mean')
).reset_index()

X_rfm = StandardScaler().fit_transform(rfm[['R','F','M']])
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_rfm)
rfm['segment'] = km.labels_

fig,axes = plt.subplots(1,2,figsize=(14,5))
for seg in range(4):
    mask = rfm['segment']==seg
    axes[0].scatter(rfm.loc[mask,'F'], rfm.loc[mask,'M'], s=10, alpha=0.5, label=f'Seg {seg}')
axes[0].set_xlabel('Frequency'); axes[0].set_ylabel('Monetary'); axes[0].legend(); axes[0].set_title('RFM 세그먼트')

seg_stats = rfm.groupby('segment').agg({'R':'mean','F':'mean','M':'mean','fraud_rate':'mean','cid':'count'})
seg_stats.columns = ['R_mean','F_mean','M_mean','Fraud_Rate','Count']
print(seg_stats.round(3))
seg_stats['Fraud_Rate'].plot(kind='bar', ax=axes[1], title='세그먼트별 이상 거래 비율')
plt.tight_layout(); plt.show()


**결과 해석**: RFM 세분화로 고빈도/고금액 고객(VIP)과 저활동 고객을 구분. 세그먼트별 이상 거래 비율 차이는 맞춤형 모니터링 전략 수립에 활용.

---
## 문제 3 풀이

다변량 이상 거래 탐지 앙상블

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report
from scipy.spatial.distance import mahalanobis
from scipy import stats as sp_stats
from datetime import datetime, timedelta

np.random.seed(42)
n=20000; nc=500; base=datetime(2025,1,1)
cids = np.random.randint(1,nc+1,n)
dates = [base+timedelta(days=int(d)) for d in np.cumsum(np.random.geometric(0.3,n))][:n]
amounts = np.random.lognormal(4,1.5,n).round(2)
cats = np.random.choice(['식료품','교통','쇼핑','외식','온라인','금융','의료'],n,p=[.25,.15,.20,.15,.10,.10,.05])
fi = np.random.choice(n,int(n*0.02),replace=False); amounts[fi]*=np.random.uniform(5,20,len(fi))
df = pd.DataFrame({'date':pd.to_datetime(dates),'cid':cids,'amount':amounts,'cat':cats,'fraud':0})
df.loc[fi,'fraud']=1

# 고객별 피처
feats = df.groupby('cid').agg(
    tx_count=('amount','count'), mean_amt=('amount','mean'), max_amt=('amount','max'),
    std_amt=('amount','std'), cat_nunique=('cat','nunique'), fraud_count=('fraud','sum')
).reset_index()
feats['std_amt'] = feats['std_amt'].fillna(0)
feats['cv'] = feats['std_amt']/(feats['mean_amt']+1)
feats['max_mean_ratio'] = feats['max_amt']/(feats['mean_amt']+1)

y_true = (feats['fraud_count']>0).astype(int)
X = feats[['tx_count','mean_amt','max_amt','std_amt','cv','max_mean_ratio','cat_nunique']].values
Xs = StandardScaler().fit_transform(X)

# Mahalanobis
mu = Xs.mean(0); Si = np.linalg.pinv(np.cov(Xs.T))
D2 = np.array([mahalanobis(x,mu,Si)**2 for x in Xs])
s_mah = MinMaxScaler().fit_transform(D2.reshape(-1,1)).flatten()

# IF
iso = IsolationForest(200, contamination=0.1, random_state=42).fit(Xs)
s_if = MinMaxScaler().fit_transform((-iso.score_samples(Xs)).reshape(-1,1)).flatten()

# LOF
lof = LocalOutlierFactor(20, contamination=0.1)
lof.fit_predict(Xs)
s_lof = MinMaxScaler().fit_transform((-lof.negative_outlier_factor_).reshape(-1,1)).flatten()

# 앙상블 (평균)
s_ens = (s_mah + s_if + s_lof) / 3
pred = (s_ens > np.percentile(s_ens, 90)).astype(int)

print(classification_report(y_true, pred, target_names=['정상','이상']))


**결과 해석**: 고객 수준 피처 엔지니어링과 앙상블 탐지로 이상 거래 패턴을 포착. CV(변동계수)와 max/mean 비율이 핵심 피처로 작용한다.

---
## 문제 4 풀이

자동 EDA 리포트 생성기

In [ ]:
import numpy as np, pandas as pd
from scipy import stats

class AutoEDAReport:
    def __init__(self, df):
        self.df = df; self.report = []
    def profile(self):
        self.report.append("=" * 60)
        self.report.append("자동 EDA 리포트")
        self.report.append("=" * 60)
        self.report.append(f"\n데이터 크기: {self.df.shape[0]} rows x {self.df.shape[1]} cols")
        self.report.append(f"메모리: {self.df.memory_usage(deep=True).sum()/1e6:.1f} MB")
        # 결측
        nulls = self.df.isnull().sum()
        if nulls.sum()>0:
            self.report.append("\n[결측값]")
            for c in nulls[nulls>0].index:
                self.report.append(f"  {c}: {nulls[c]} ({nulls[c]/len(self.df):.1%})")
        else:
            self.report.append("\n[결측값] 없음")
        # 수치형 프로파일
        self.report.append("\n[수치형 변수 프로파일]")
        for c in self.df.select_dtypes(include=[np.number]).columns:
            s = self.df[c]
            sk = s.skew(); ku = s.kurtosis()
            _, norm_p = stats.normaltest(s.dropna()) if len(s.dropna())>20 else (0,1)
            outlier_iqr = ((s<s.quantile(0.25)-1.5*(s.quantile(0.75)-s.quantile(0.25))) |
                           (s>s.quantile(0.75)+1.5*(s.quantile(0.75)-s.quantile(0.25)))).sum()
            self.report.append(f"  {c}: mean={s.mean():.2f}, std={s.std():.2f}, skew={sk:.2f}, "
                              f"kurtosis={ku:.2f}, outliers(IQR)={outlier_iqr}, normal_p={norm_p:.4f}")
        # 범주형
        self.report.append("\n[범주형 변수]")
        for c in self.df.select_dtypes(include=['object','category']).columns:
            n_unique = self.df[c].nunique()
            top = self.df[c].value_counts().head(3)
            self.report.append(f"  {c}: {n_unique} categories, top3={dict(top)}")
        return '\n'.join(self.report)

# 테스트
np.random.seed(42)
df_test = pd.DataFrame({
    'amount': np.random.lognormal(4,1.5,1000),
    'count': np.random.poisson(5,1000),
    'cat': np.random.choice(['A','B','C'],1000),
    'score': np.random.normal(50,10,1000),
})
df_test.loc[np.random.choice(1000,50), 'score'] = np.nan

report = AutoEDAReport(df_test)
print(report.profile())


**결과 해석**: 자동 EDA 리포트는 수동 탐색 시간을 줄이고 데이터 품질 문제를 체계적으로 파악하는 출발점이다. 프로덕션에서는 이를 스케줄링하여 정기 모니터링에 활용한다.

---
## 확장 토론

### 금융 EDA 체크리스트

1. 시간 패턴: 주기성, 추세, 이상 시점
2. 금액 분포: Benford, 이상치, 계절성
3. 고객 세분화: RFM + 행동 패턴
4. 이상 탐지: 다변량 앙상블
5. 자동화: 리포트 + 알림 파이프라인